In [4]:
import yaml

with open("./Indoor--1/data.yaml", "r") as f:
    data_config = yaml.safe_load(f)

train_list = data_config["train"]  # This should be a list of paths or image-label pairs


In [7]:
from torch.utils.data import Dataset
from PIL import Image
import os

class ImageDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list  # Should be list of (image_path, label_path) or just image paths
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, label = self.data_list[idx] if isinstance(self.data_list[idx], (list, tuple)) else (self.data_list[idx], None)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [8]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((640, 640)),  # or any target input shape
    transforms.ToTensor(),
])

dataset = ImageDataset(train_list, transform=transform)
calibration_loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False)


In [9]:
import nncf
import torch

calibration_loader = torch.utils.data.DataLoader(...)

def transform_fn(data_item):
    images, _ = data_item
    return images

calibration_dataset = nncf.Dataset(calibration_loader, transform_fn)

In [14]:
from ultralytics import YOLO

model = YOLO("best.pt")  # This returns a wrapper, not nn.Module


WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\vidha\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [16]:
pytorch_model = model.model.eval()  # This is the raw nn.Module
